# Imports and data loading

In [19]:
import glob
import os
import re

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import screed
import seaborn as sns
from joblib import Parallel, delayed
from scipy.special import logsumexp
from tqdm import tqdm
from tqdm.auto import tqdm

import sourmash

tqdm.pandas()


from kmer_utils import get_encoded_kmer_hashvals

%load_ext autoreload
%autoreload 2

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [20]:
outdir = "/Users/olgabot/botryllus/adhoc-analysis/2022-apr--gather-botryllus-in-human-mouse-with-kmers/"
# ! mkdir $outdir

## Data locations

In [28]:
botryllus_dir = "/Users/olgabot/botryllus/data/botryllus-proteins/"

human_gencode_dir = "/Users/olgabot/botryllus/data/gencode/v38/"
mouse_gencode_dir = "/Users/olgabot/botryllus/data/gencode/M28/"

## set signature files

human_sigfile = os.path.join(
    human_gencode_dir, "gencode.v38.basic.annotation.protein.fa.hp.k24.scale5.sig"
)
mouse_sigfile = os.path.join(
    mouse_gencode_dir, "gencode.vM28.basic.annotation.protein.fa.hp.k24.scale5.sig"
)


botryllus_sigfile = os.path.join(botryllus_dir, "Bs_proteins.fa.hp.k24.scale5.sig")


human_db = sourmash.load_file_as_index(human_sigfile)

mouse_db = sourmash.load_file_as_index(mouse_sigfile)

dbs = {"human": human_db, "mouse": mouse_db}

In [45]:
human_sigfile

'/Users/olgabot/botryllus/data/gencode/v38/gencode.v38.basic.annotation.protein.fa.hp.k24.scale5.sig'

### Read in human-mouse homologs

In [35]:
human_mouse_homologs = pd.read_csv("HOM_MouseHumanSequence.rpt", sep="\t")
print(human_mouse_homologs.shape)
human_mouse_homologs.head()

(43117, 13)


,DB Class Key,Common Organism Name,NCBI Taxon ID,Symbol,EntrezGene ID,Mouse MGI ID,HGNC ID,OMIM Gene ID,Genetic Location,"Genomic Coordinates (mouse: , human: )",Nucleotide RefSeq IDs,Protein RefSeq IDs,SWISS_PROT IDs
0,39806032,"mouse, laboratory",10090,Gdnf,14573,MGI:107430,NaN,NaN,Chr15 3.8 cM,Chr15:7840327-7867056(+),"NM_010275,NM_001301333,NM_001301357,NM_001301332","NP_001288262,NP_034405,NP_001288286,NP_001288261",P48540
1,39806032,human,9606,GDNF,2668,NaN,HGNC:4232,OMIM:600837,Chr5 p13.2,Chr5:37812677-37840044(-),"NM_199231,NM_001278098,NM_001190469,NM_000514,...","NP_001177398,NP_000505,NP_001177397,XP_0168648...",P39905
2,39806033,"mouse, laboratory",10090,Npy4r,19065,MGI:105374,NaN,NaN,Chr14 20.8 cM,Chr14:33867603-33874376(-),NM_008919,NP_032945,Q61041
3,39806033,human,9606,NPY4R,5540,NaN,HGNC:9329,OMIM:601790,Chr10 q11.22,Chr10:46458551-46470668(-),"NM_001278794,NM_005972","NP_005963,XP_011538238,XP_011538239,XP_0168718...",NaN
4,39806034,"mouse, laboratory",10090,Evx2,14029,MGI:95462,NaN,NaN,Chr2 44.13 cM,Chr2:74483335-74489901(-),"XM_006498728,XM_006498729,NM_007967","XP_006498792,NP_031993,XP_006498791",P49749


# Run "gather" to identify shared k-mers from bortyllus to human

## Load human, mouse, botryllus fastas to read in sequences to write matching k-mers

In [36]:
def fasta_to_dict(fasta_filename):
    sequences = {}
    with screed.open(fasta_filename) as records:
        for record in records:
            sequences[record["name"]] = record["sequence"]

    return sequences

In [37]:
botryllus_sequences = fasta_to_dict(os.path.join(botryllus_dir, "Bs_proteins.fa.gz"))
n_botryllus_seqs = len(botryllus_sequences)
n_botryllus_seqs

72617

In [38]:
fastas = {
    "human": os.path.join(human_gencode_dir, "gencode.v38.basic.annotation.protein.fa"),
    "mouse": os.path.join(
        mouse_gencode_dir, "gencode.vM28.basic.annotation.protein.fa"
    ),
}
fastas

{'human': '/Users/olgabot/botryllus/data/gencode/v38/gencode.v38.basic.annotation.protein.fa',
 'mouse': '/Users/olgabot/botryllus/data/gencode/M28/gencode.vM28.basic.annotation.protein.fa'}

In [39]:
%time sequences = {k: fasta_to_dict(fasta) for k, fasta in fastas.items()}
for species, seqs in sequences.items():
    print(f"{species}, {len(seqs)}")

CPU times: user 769 ms, sys: 102 ms, total: 871 ms
Wall time: 921 ms
human, 61543
mouse, 46083


## Move code into functions for a separate file to run on all botryllus sequences

### Load botryllus protein signatures

In [40]:
botryllus_sigs = sourmash.load_file_as_signatures(botryllus_sigfile)

### Iterate over all botryllus signatures

In [43]:
! sourmash scripts fastgather --help

/Users/olgabot/.zshenv:.:1: no such file or directory: /Users/olgabot/.cargo/env
 scripts: error: argument subcmd: invalid choice: 'fastgather' (choose from )


In [42]:
from single_gather_multi_db import single_gather_multi_db

total = n_botryllus_seqs

for query_sig in tqdm(botryllus_sigs, total=total):
    query_seq = botryllus_sequences[query_sig.name]

    single_gather_multi_db(
        query_sig, query_seq, dbs, sequences, human_mouse_homologs, outdir, force=False
    )

g4.t1 frame:1 had no matches                                                                                                                                                                     | 0/72617 [00:00<?, ?it/s]
g20.t1 frame:3 had no matches                                                                                                                                                        | 1/72617 [00:06<127:26:44,  6.32s/it]
g23.t1 frame:1 had no matches                                                                                                                                                        | 17/72617 [00:12<12:46:43,  1.58it/s]
g25.t1 frame:1 had no matches                                                                                                                                                        | 20/72617 [00:18<18:23:01,  1.10it/s]
g29.t1 frame:1 had no matches                                                                                           

In [ ]:
csvs_created = glob.glob(os.path.join(outdir, "*.csv"))
n_csvs_created = len(csvs_created)
n_csvs_created

# Concatenate gather results

## Set output directory of `gather`

In [ ]:
outdir = "/Users/olgabot/botryllus/adhoc-analysis/2022-apr--gather-botryllus-in-human-mouse-with-kmers/"
# ! mkdir $outdir

In [ ]:
! ls -lha $outdir | head

In [ ]:
df = pd.read_csv(os.path.join(outdir, "g34-dot-t1_frame-colon-1.csv"))
df.head()

In [ ]:
%%time

csvs = glob.glob(os.path.join(outdir, "*.csv"))
dfs = []

for csv in tqdm(csvs):
    df = pd.read_csv(csv)
    # Convert hashval to string
    df["hashval"] = df["hashval"].astype(str)
    df["transcript_id"] = df["name_found"].str.split().str[0]
    dfs.append(df)
gather_results = pd.concat(dfs)
print(gather_results.shape)
gather_results.head()

In [ ]:
parquet = os.path.join(outdir, "botryllus_gather_mouse_human_results.parquet")

In [ ]:
%%time
gather_results.to_parquet(parquet)

## Read gather results

In [ ]:
%%time

gather_results = pd.read_parquet(
    os.path.join(outdir, "botryllus_gather_mouse_human_results.parquet")
)

## Show top k-mers

In [ ]:
hashval_to_hp_kmer = pd.Series(
    dict(zip(gather_results["hashval"], gather_results["kmer_hp"]))
)
# hashval_to_hp_kmer.index = hashval_to_hp_kmer.index.astype(int)
print(hashval_to_hp_kmer.shape)
hashval_to_hp_kmer.drop_duplicates()
print(hashval_to_hp_kmer.shape)

In [ ]:
hashval_to_hp_kmer

In [ ]:
hp_kmer_counts = gather_results.kmer_hp.value_counts()
hp_kmer_counts.head()

In [ ]:
len(hp_kmer_counts)

In [ ]:
hp_kmer_counts.tail()

### Subset to only human gather results

In [ ]:
gather_results_human = gather_results.query('species == "human"')
gather_results_human.head()

### Delete original `gather_results` for memory use

In [ ]:
del gather_results

# Get signature files

## Create non-singleton sourmash sketch

In [ ]:
human_fasta = os.path.join(human_gencode_dir, "gencode.v38.basic.annotation.protein.fa")
human_sigfile_aggregated = os.path.join(
    human_gencode_dir,
    "gencode.v38.basic.annotation.protein.fa.hp.k24.scale5.aggregated.sig",
)
# ! time sourmash sketch protein --output $human_sigfile_aggregated --hp -p k=24,scaled=5,abund $human_fasta

In [ ]:
human_sig = sourmash.load_one_signature(human_sigfile_aggregated)

### Read botryllus kmer hashes

In [ ]:
def read_hash_csv(csv):
    df = pd.read_csv(csv)

    # Force hashval to be strings to avoid overflow/underflow errors
    df["hashval"] = df["hashval"].astype(str)
    df = df.set_index(["hashval"])
    return df

In [ ]:
csv = os.path.join(botryllus_dir, "Bs_proteins.fa.hp.k24.scale5.aggregated.kmers.csv")
botryllus_kmer_hashes = read_hash_csv(csv)
# botryllus_kmer_hashes.index = botryllus_kmer_hashes.index.astype(str)
botryllus_kmer_hashes

# Get signature files

In [ ]:
botryllus_dir = "/Users/olgabot/botryllus/data/botryllus-proteins/"

human_gencode_dir = "/Users/olgabot/botryllus/data/gencode/v38/"
mouse_gencode_dir = "/Users/olgabot/botryllus/data/gencode/M28/"

## set signature files

human_sigfile = os.path.join(
    human_gencode_dir, "gencode.v38.basic.annotation.protein.fa.hp.k24.scale5.sig"
)
mouse_sigfile = os.path.join(
    mouse_gencode_dir, "gencode.vM28.basic.annotation.protein.fa.hp.k24.scale5.sig"
)

botryllus_sigfile = os.path.join(botryllus_dir, "Bs_proteins.fa.hp.k24.scale5.sig")

## Create non-singleton sourmash sketch

In [ ]:
human_fasta = os.path.join(human_gencode_dir, "gencode.v38.basic.annotation.protein.fa")
human_sigfile_aggregated = os.path.join(
    human_gencode_dir,
    "gencode.v38.basic.annotation.protein.fa.hp.k24.scale5.aggregated.sig",
)
# ! time sourmash sketch protein --output $human_sigfile_aggregated --hp -p k=24,scaled=5,abund $human_fasta

In [ ]:
human_sig = sourmash.load_one_signature(human_sigfile_aggregated)

### Read botryllus kmer hashes

In [ ]:
def read_hash_csv(csv):
    df = pd.read_csv(csv)

    # Force hashval to be strings to avoid overflow/underflow errors
    df["hashval"] = df["hashval"].astype(str)
    df = df.set_index(["hashval"])
    return df

In [ ]:
csv = os.path.join(botryllus_dir, "Bs_proteins.fa.hp.k24.scale5.aggregated.kmers.csv")
botryllus_kmer_hashes = read_hash_csv(csv)
# botryllus_kmer_hashes.index = botryllus_kmer_hashes.index.astype(str)
botryllus_kmer_hashes

# Make pandas series of hashes

In [ ]:
%%time
human_minhashes = pd.Series(human_sig.minhash.hashes, name="human")
human_minhashes.index = human_minhashes.index.astype(str)

In [ ]:
human_minhashes.describe()

In [ ]:
botryllus_sigfile_aggregated = os.path.join(
    botryllus_dir, "Bs_proteins.fa.hp.k24.scale5.aggregated.sig"
)
botryllus_sig = sourmash.load_one_signature(botryllus_sigfile_aggregated)

In [ ]:
%%time
botryllus_minhashes = pd.Series(botryllus_sig.minhash.hashes, name="botryllus")
botryllus_minhashes.index = botryllus_minhashes.index.astype(str)

botryllus_minhashes.describe()

# Compute probability of overlap given underlying k-mer distribution

In [ ]:
gather_results_subset = gather_results_human.head(100)
len(gather_results_subset.groupby(["name_query", "name_found"]))

## Compute frequencies of each minhash

In [ ]:
botryllus_minhashes_freq = botryllus_minhashes / botryllus_minhashes.sum()
human_minhashes_freq = human_minhashes / human_minhashes.sum()

In [ ]:
botryllus_minhashes_freq_log = np.log(botryllus_minhashes_freq)
human_minhashes_freq_log = np.log(botryllus_minhashes_freq)

## Function to compute probability of overlap

In [ ]:
def get_prob_overlap_log(df, log_freq_a, log_freq_b):
    # import pdb; pdb.set_trace()
    hashes = df["hashval"].values

    # Not all hashes are present for some reason -> take the intersection for now while I figure that bug out
    observed_hashes = log_freq_a.index.intersection(
        log_freq_b.index.intersection(hashes)
    )
    log_freq_a_subset = log_freq_a[observed_hashes]
    log_freq_b_subset = log_freq_b[observed_hashes]
    prob_overlap = np.logaddexp(log_freq_a_subset, log_freq_b_subset)
    return prob_overlap


def get_prob_overlap(df, freq_a, freq_b):
    # import pdb; pdb.set_trace()
    hashes = df["hashval"].values

    # Not all hashes are present for some reason -> take the intersection for now while I figure that bug out
    # Was something done at scaled=10 maybe? and then this is scaled=5 which has half the k-mers?
    observed_hashes = freq_a.index.intersection(freq_b.index.intersection(hashes))
    # observed_hashes = hashes
    freq_a_subset = freq_a[observed_hashes]
    freq_b_subset = freq_b[observed_hashes]
    prob_overlap = freq_a_subset.multiply(freq_b_subset).sum()
    return prob_overlap


# Test on a small subset
gather_results_subset = gather_results_human.head(100)
len(gather_results_subset.groupby(["name_query", "name_found"]))
gather_results_subset_prob_overlap = gather_results_subset.groupby(
    ["name_query", "name_found"]
).apply(lambda x: get_prob_overlap(x, botryllus_minhashes_freq, human_minhashes_freq))
gather_results_subset_prob_overlap.describe()

In [ ]:
assert gather_results_subset_prob_overlap.describe().to_dict() == {
    "count": 27.0,
    "mean": 2.2792966660606903e-11,
    "std": 2.6425894660527337e-11,
    "min": 0.0,
    "25%": 1.8966495671697554e-12,
    "50%": 7.481228848280702e-12,
    "75%": 4.8680672224023717e-11,
    "max": 6.559246419795403e-11,
}

### This step could probably be precomputed to make it easier

In [ ]:
human_botryllus_kmer_probs = gather_results_human.groupby(
    ["name_query", "name_found"]
).progress_apply(
    lambda x: get_prob_overlap(x, botryllus_minhashes_freq, human_minhashes_freq)
)
human_botryllus_kmer_probs

In [ ]:
assert human_botryllus_kmer_probs.describe().to_dict() == {
    "count": 577598.0,
    "mean": 5.596240440926977e-08,
    "std": 1.7350764402400693e-07,
    "min": 0.0,
    "25%": 1.3171177549789967e-12,
    "50%": 4.05672268533531e-12,
    "75%": 4.2674615261319495e-11,
    "max": 6.152694316601546e-07,
}

### Do simple bonferroni correction of probabilities

In [ ]:
n_comparisons = len(human_botryllus_kmer_probs)
n_comparisons

In [ ]:
human_botryllus_kmer_probs_adjusted = human_botryllus_kmer_probs * n_comparisons
assert human_botryllus_kmer_probs_adjusted.describe().to_dict() == {
    "count": 577598.0,
    "mean": 0.03232377286198539,
    "std": 0.10021766817297832,
    "min": 0.0,
    "25%": 7.607645810403585e-07,
    "50%": 2.3431549096043043e-06,
    "75%": 2.4648772425707616e-05,
    "max": 0.355378393188042,
}

In [ ]:
sns.displot(np.log10(human_botryllus_kmer_probs_adjusted))

## Join with gather results

### Prep data to be joined

In [ ]:
groupby = ["name_query", "name_found"]
human_botryllus_kmer_probs_adjusted.name = "p_value_adjusted"
human_botryllus_kmer_probs.name = "p_value"

### Do the join

In [ ]:
%%time

gather_results_human_with_probs = gather_results_human.join(
    human_botryllus_kmer_probs_adjusted, on=groupby
).join(human_botryllus_kmer_probs, on=groupby)
gather_results_human_with_probs["p_value_adjusted_log10"] = np.log10(
    gather_results_human_with_probs["p_value_adjusted"]
)
gather_results_human_with_probs["p_value_log10"] = np.log10(
    gather_results_human_with_probs["p_value"]
)


gather_results_human_with_probs.head()

In [ ]:
gather_results_human_with_probs["containment_adjusted"] = (
    gather_results_human_with_probs.containment.divide(
        gather_results_human_with_probs.p_value_adjusted
    )
)
gather_results_human_with_probs["containment_adjusted_log10"] = np.log10(
    gather_results_human_with_probs["containment_adjusted"]
)


gather_results_human_with_probs.head()

In [ ]:
float("inf")

In [ ]:
assert gather_results_human_with_probs[
    "p_value_adjusted_log10"
].describe().dropna().to_dict() == {
    "count": 10894207.0,
    "mean": -float("inf"),
    "min": -float("inf"),
    "25%": -4.268471162554706,
    "50%": -0.46422337482473364,
    "75%": -0.45812018563930756,
    "max": -0.4493089805950437,
}

assert gather_results_human_with_probs[
    "p_value_log10"
].describe().dropna().to_dict() == {
    "count": 10894207.0,
    "mean": -float("inf"),
    "min": -float("inf"),
    "25%": -10.030096843323927,
    "50%": -6.225849055593955,
    "75%": -6.2197458664085294,
    "max": -6.2109346613642655,
}


assert gather_results_human_with_probs[
    "containment_adjusted"
].describe().dropna().to_dict() == {
    "count": 10894207.0,
    "mean": float("inf"),
    # "std": np.nan,
    "min": 0.006075402734956316,
    "25%": 0.06098912318089084,
    "50%": 0.1460184238113183,
    "75%": 648.2360256675821,
    "max": float("inf"),
}

assert gather_results_human_with_probs[
    "containment_adjusted_log10"
].describe().dropna().to_dict() == {
    "count": 10894207.0,
    "mean": float("inf"),
    # "std": np.nan,
    "min": -2.216424927643745,
    "25%": -1.2147476102959802,
    "50%": -0.8355923438403171,
    "75%": 2.8117331632495297,
    "max": float("inf"),
}

In [ ]:
gather_results_human_with_probs_dedup = gather_results_human_with_probs.drop_duplicates(
    ["name_query", "name_found"]
).drop(
    [
        "i_query",
        "i_found",
        "found_i",
    ],
    axis=1,
)
gather_results_human_with_probs_dedup

### Write to file

In [ ]:
%%time
gather_results_human_with_probs_dedup.to_parquet(
    os.path.join(
        outdir, "botryllus_gather_human_results_adjusted_by_kmer_probabilities.parquet"
    )
)

In [ ]:
sns.displot(gather_results_human_with_probs_dedup.containment)

In [ ]:
sns.displot(gather_results_human_with_probs_dedup.containment_adjusted_log10)

## What is the bump at log(containment_adjusted) = 0 ? These are false positives right?

In [ ]:
"p_value_adjusted	p_value	containment_adjusted	containment_adjusted_log10".split()

In [ ]:
cols = [
    "kmer_query",
    "kmer_found",
    "kmer_hp",
    # "hashval",
    "name_query",
    "symbol",
    "n_kmers",
    "intersect_bp",
    "p_value_adjusted",
    # "p_value",
    "containment",
    "containment_adjusted",
    "containment_adjusted_log10",
]

gather_results_human_with_probs_dedup.query("containment_adjusted < 1")[cols]

In [ ]:
gather_results_human_with_probs.query("containment_adjusted < 1").kmer_hp.value_counts()

### Check for TTN -> Likely false positive

In [ ]:
gather_results_human_with_probs_dedup.query('symbol == "TTN"')

## Check BHF and HLA genes

In [ ]:
bhf = gather_results_human_with_probs.query('name_query == "BHF"')
bhf = bhf.reset_index(drop=True)
bhf

## Remove duplicates

In [ ]:
bhf_per_position = bhf.drop_duplicates(["i_query", "symbol"]).sort_values("i_query")
bhf_per_position.head()

## Stitch together k-mers

In [ ]:
def single_stitch_together_kmers(kmer_series):
    # stitched = ''
    for i, kmer in enumerate(kmer_series):
        if i == 0:
            stitched = kmer
        else:
            stitched += kmer[-1]
    return stitched


def stitch_together_species_hp_kmers(
    df,
    kmer_query="kmer_query",
    kmer_hp="kmer_hp",
    kmer_found="kmer_found",
    i_query="i_query",
):
    df = df.sort_values(i_query)
    query_stitched = single_stitch_together_kmers(df[kmer_query])
    hp_stitched = single_stitch_together_kmers(df[kmer_hp])
    found_stitched = single_stitch_together_kmers(df[kmer_found])
    start = df.i_query.min()
    end = start + len(found_stitched)
    to_print = (
        f"botryllus: {query_stitched}\nhp: {hp_stitched}\nhuman: {found_stitched}"
    )
    return pd.Series([start, end, to_print], index=["start", "end", "matches"])


bhf_per_position_matches = (
    bhf_per_position.groupby("symbol")
    .apply(stitch_together_species_hp_kmers)
    .sort_values(["start", "end"])
)
bhf_per_position_matches

In [ ]:
bhf_per_position_matches.to_clipboard()

In [ ]:
for gene, row in bhf_per_position_matches.iterrows():
    print(f"\n{gene} {row.start}-{row.end} aa")
    print(row.matches)